# Assemble datasets from simulations
Combine data from simulations of different network architectures

In [1]:
import numpy as np
import pandas as pd
import os
import pickle
from tqdm import tqdm
from joblib import Parallel, delayed
import re

from stoch_sim_model import *

In [2]:
# Set parameters
sim_kind = 'agent'
reg_model = ''
runs = '-5-'
comment = "acute_all-sparse-reg"

d = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/'

sim_sum_list = []
parameters_nets = []
prim_diff_bias_list = []
#sec_diff_bias_list = []
cell_series_list = []
# lineage_diff_nets = []

In [3]:
# Figure out which jobs didn't run:
d_rerun = '/gscratch/scrubbed/oukogu/infoimmune/sim_output/no_cell_var/raw/'
run_list = [int(re.search('sim_batch_(.*?)\.', f).group(1)) for f in os.listdir(d_rerun) if 'sim_batch' in f and comment in f and runs in f]
out = [str(x) for x in [k for k in np.arange(0, 1407)] if x not in run_list]
print(len(out))
print(' '.join((out)))

7
563 564 566 637 1342 1347 1350


In [4]:
num_cpu = 150
file_list = [f for f in os.listdir(os.path.join(d, "raw")) if runs in f and comment in f and 'sim_batch' in f]
num_files = len(file_list)

def import_dict_func(f,d):
    
    file_path = os.path.join(os.path.join(d, "raw"), f)
    with open(file_path, 'rb') as filename:  
        import_dict = pickle.load(filename)

    parameters = np.array(import_dict["parameters"])
    sim_sum = np.array(import_dict["summary_stats"])

    out = np.hstack((parameters, sim_sum))

    return out

# create dataframe of infection response statistics
var_names = np.concatenate((param_names_for_df, stat_names_for_df))
mean_df = pd.DataFrame(np.vstack(Parallel(n_jobs = num_cpu, batch_size = max(int(num_files/num_cpu),1))(delayed(import_dict_func)(f = file_name, d = d) 
                                                                                                        for file_name in file_list)), 
                       columns = [i for i in var_names]).groupby(Na_reg + NE_reg + EM_reg + EE_reg + vir_vars, as_index=False).mean()

In [5]:
# Save datasets
### (1) Summary stats
mean_df.to_pickle(os.path.join(d, "raw", "stacked_data"+runs+"runs"+'-'+comment)+'.pkl')
del mean_df